# Facebook Hashtag Photo Scraper

Notebook ini dibuat untuk mengambil **foto pertama** dari posting Facebook dengan hashtag `#wisatalampungselatan`, lalu menyimpan metadata ke CSV dengan kolom:

- `file_name`
- `caption`
- `like`
- `date`
- `location`

Catatan penting:

- Notebook ini memakai **Selenium** dan mengandalkan tampilan Facebook yang bisa berubah sewaktu-waktu.
- Jalankan hanya untuk post yang memang bisa Anda akses secara sah.
- Untuk simpan foto ke Google Drive, paling mudah arahkan folder output ke **folder Google Drive Desktop yang tersinkron di komputer Anda**.
- Post yang berupa **video akan di-skip**.


In [11]:
import importlib.util
import subprocess
import sys

required_packages = {
    "selenium": "selenium",
    "pandas": "pandas",
    "requests": "requests",
}

missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Menginstal package yang belum ada:", ", ".join(missing_packages))
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade-strategy",
        "only-if-needed",
        *missing_packages,
    ])
else:
    print("Semua package utama sudah tersedia, install dilewati.")


Semua package utama sudah tersedia, install dilewati.


In [12]:
import hashlib
import json
import re
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import parse_qs, quote, urlencode, urlparse, urlunparse

import pandas as pd
import requests
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException, WebDriverException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.selenium_manager import SeleniumManager
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait


# =========================
# KONFIGURASI UTAMA
# =========================
HASHTAG = "wisatalampungselatan"
MAX_SAVED_POSTS = 20
COLLECT_LINK_TARGET = max(MAX_SAVED_POSTS * 8, 100)
SCROLL_ROUNDS = 35
SCROLL_PAUSE_SECONDS = 4
POST_LOAD_WAIT_SECONDS = 4
HEADLESS = False

# Isi jika Anda ingin foto langsung masuk ke folder Google Drive yang sinkron di PC.
# Contoh:
# GOOGLE_DRIVE_SYNC_FOLDER = r"C:\\Users\\NamaUser\\My Drive\\facebook_wisata_lampung_selatan"
GOOGLE_DRIVE_SYNC_FOLDER = r""

LOCAL_OUTPUT_DIR = Path.cwd() / "output"
IMAGE_OUTPUT_DIR = Path(GOOGLE_DRIVE_SYNC_FOLDER) if GOOGLE_DRIVE_SYNC_FOLDER else (LOCAL_OUTPUT_DIR / "photos")
CSV_OUTPUT_PATH = LOCAL_OUTPUT_DIR / "facebook_hashtag_posts.csv"
LINK_DEBUG_OUTPUT_PATH = LOCAL_OUTPUT_DIR / "collected_post_links.txt"
REQUEST_TIMEOUT = 60
SELENIUM_CACHE_DIR = LOCAL_OUTPUT_DIR / "selenium_cache"

# Optional: isi kalau mau pakai browser profile khusus.
CHROME_USER_DATA_DIR = r""
CHROME_PROFILE_DIRECTORY = r""
CHROME_BINARY_PATH = r""
CHROMEDRIVER_PATH = r""

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/135.0.0.0 Safari/537.36"
)

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SELENIUM_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Folder foto   : {IMAGE_OUTPUT_DIR}")
print(f"File CSV      : {CSV_OUTPUT_PATH}")
print(f"Hashtag target: #{HASHTAG}")

Folder foto   : c:\Users\elsae\Downloads\Scrapp\output\photos
File CSV      : c:\Users\elsae\Downloads\Scrapp\output\facebook_hashtag_posts.csv
Hashtag target: #wisatalampungselatan


In [ ]:
def normalize_space(value):
    if value is None:
        return None
    value = re.sub(r"\s+", " ", str(value)).strip()
    return value or None


def find_first_existing_path(candidates):
    for candidate in candidates:
        if not candidate:
            continue
        candidate_path = Path(candidate)
        if candidate_path.exists():
            return str(candidate_path)
    return None


def resolve_chrome_browser_path():
    browser_candidates = [
        CHROME_BINARY_PATH,
        r"C:\Program Files\Google\Chrome\Application\chrome.exe",
        r"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe",
    ]
    browser_path = find_first_existing_path(browser_candidates)
    if not browser_path:
        raise FileNotFoundError(
            "Chrome tidak ditemukan. Isi CHROME_BINARY_PATH dengan lokasi chrome.exe di komputer Anda."
        )
    return browser_path


def resolve_chromedriver_path(browser_path):
    explicit_driver = find_first_existing_path([CHROMEDRIVER_PATH])
    if explicit_driver:
        return explicit_driver

    selenium_manager_binary = str(SeleniumManager._get_binary())
    command = [
        selenium_manager_binary,
        "--browser",
        "chrome",
        "--browser-path",
        browser_path,
        "--language-binding",
        "python",
        "--output",
        "json",
        "--cache-path",
        str(SELENIUM_CACHE_DIR),
        "--avoid-stats",
    ]

    completed = subprocess.run(command, capture_output=True, text=True)
    if completed.returncode != 0:
        raise RuntimeError(
            "Gagal mendapatkan chromedriver otomatis via Selenium Manager.\n"
            f"STDOUT: {completed.stdout}\nSTDERR: {completed.stderr}"
        )

    payload = json.loads(completed.stdout or "{}")
    result = payload.get("result", {})
    driver_path = result.get("driver_path")
    if not driver_path:
        raise RuntimeError(f"Driver path tidak ditemukan di output Selenium Manager: {payload}")

    return driver_path


def build_driver():
    browser_path = resolve_chrome_browser_path()
    driver_path = resolve_chromedriver_path(browser_path)
    service = Service(executable_path=driver_path)
    chrome_options = Options()
    chrome_options.binary_location = browser_path
    if HEADLESS:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument(f"--user-agent={USER_AGENT}")
    chrome_options.add_argument("--disable-notifications")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--lang=en-US")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)

    if CHROME_USER_DATA_DIR:
        chrome_options.add_argument(f"--user-data-dir={CHROME_USER_DATA_DIR}")
    if CHROME_PROFILE_DIRECTORY:
        chrome_options.add_argument(f"--profile-directory={CHROME_PROFILE_DIRECTORY}")

    driver = webdriver.Chrome(service=service, options=chrome_options)
    driver.set_page_load_timeout(120)
    return driver


def wait_for_manual_login(driver):
    driver.get("https://www.facebook.com/")
    print("Login ke Facebook di browser yang terbuka.")
    print("Setelah login selesai dan halaman Facebook sudah siap, kembali ke notebook lalu tekan Enter.")
    input()


def build_requests_session_from_driver(driver):
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})
    for cookie in driver.get_cookies():
        session.cookies.set(
            cookie["name"],
            cookie["value"],
            domain=cookie.get("domain"),
            path=cookie.get("path", "/"),
        )
    return session


def normalize_facebook_url(url):
    if not url:
        return None

    parsed = urlparse(url)
    if "facebook.com" not in parsed.netloc:
        return None

    path = parsed.path.rstrip("/") or "/"
    query = parse_qs(parsed.query)

    if any(token in path for token in ["/reel/", "/watch/", "/videos/"]):
        return None

    if path in {"/photo", "/photo/", "/photo.php"}:
        keep = {}
        for key in ["fbid", "set"]:
            if query.get(key):
                keep[key] = query[key][0]
        if not keep.get("fbid"):
            return None
        clean_query = urlencode(keep)
        return urlunparse(("https", "www.facebook.com", "/photo", "", clean_query, ""))

    if path.endswith("/permalink.php") or path.endswith("/story.php"):
        keep = {}
        for key in ["story_fbid", "id"]:
            if query.get(key):
                keep[key] = query[key][0]
        if not keep:
            return None
        clean_query = urlencode(keep)
        return urlunparse(("https", "www.facebook.com", path, "", clean_query, ""))

    if "/posts/" in path or "/groups/" in path:
        return urlunparse(("https", "www.facebook.com", path, "", "", ""))

    if query.get("fbid"):
        clean_query = urlencode({"fbid": query["fbid"][0]})
        return urlunparse(("https", "www.facebook.com", path, "", clean_query, ""))

    return None


def is_candidate_post_url(url):
    clean_url = normalize_facebook_url(url)
    if not clean_url:
        return False

    parsed = urlparse(clean_url)
    path = parsed.path.rstrip("/") or "/"
    query = parse_qs(parsed.query)

    if "/posts/" in path or path.endswith("/permalink.php") or path.endswith("/story.php"):
        return True
    if path in {"/photo", "/photo.php"} and query.get("fbid"):
        return True
    if "/groups/" in path and ("/posts/" in path or query.get("multi_permalinks")):
        return True
    return False


def get_post_identity_key(url):
    clean_url = normalize_facebook_url(url)
    if not clean_url:
        return None

    parsed = urlparse(clean_url)
    path = parsed.path.rstrip("/") or "/"
    query = parse_qs(parsed.query)

    for key in ["story_fbid", "fbid"]:
        values = query.get(key)
        if values and values[0]:
            return f"{key}:{values[0]}"

    if "/posts/" in path:
        token = path.split("/posts/", 1)[1].split("/", 1)[0]
        token = normalize_space(token)
        if token:
            return f"posts:{token}"

    if "/groups/" in path:
        for key in ["multi_permalinks", "story_fbid", "fbid"]:
            values = query.get(key)
            if values and values[0]:
                return f"groups:{values[0]}"

    clean_pairs = sorted((key, values[0]) for key, values in query.items() if values)
    if clean_pairs:
        return f"url:{path.lower()}?{urlencode(clean_pairs)}"
    return f"url:{path.lower()}"


def extract_post_identity_key(driver, fallback_url=None):
    candidates = [driver.current_url, fallback_url]
    for candidate_url in candidates:
        key = get_post_identity_key(candidate_url)
        if key:
            return key

    html = driver.page_source or ""
    patterns = [
        r'"story_fbid":\"?(\d+)\"?',
        r'"mf_story_key":\"?(\d+)\"?',
        r'"post_id":\"?(\d+)\"?',
        r'"ft_ent_identifier":\"?(\d+)\"?',
        r'"legacy_fbid":\"?(\d+)\"?',
    ]
    for pattern in patterns:
        match = re.search(pattern, html)
        if match:
            return f"page:{match.group(1)}"

    return None


def get_search_urls(hashtag):
    hashtag_value = hashtag.lstrip("#")
    encoded_hashtag = quote(hashtag_value)
    encoded_query = quote(f"#{hashtag_value}")
    return [
        f"https://www.facebook.com/hashtag/{encoded_hashtag}",
        f"https://www.facebook.com/search/posts/?q={encoded_query}",
        f"https://www.facebook.com/search/top/?q={encoded_query}",
    ]


def extract_candidate_hrefs(driver):
    hrefs = driver.execute_script(
        """
        return Array.from(document.querySelectorAll('a[href]'))
            .map(anchor => anchor.href)
            .filter(Boolean);
        """
    )

    html = driver.page_source
    regex_hrefs = re.findall(r'https://www\\.facebook\\.com/[^\"\'\s<>]+', html)
    return list(dict.fromkeys((hrefs or []) + regex_hrefs))


def get_meta_content(driver, css_selectors):
    for selector in css_selectors:
        try:
            element = driver.find_element(By.CSS_SELECTOR, selector)
            content = normalize_space(element.get_attribute("content"))
            if content:
                return content
        except NoSuchElementException:
            continue
    return None


def collect_post_links(driver, hashtag, target_count, scroll_rounds=20, pause_seconds=3):
    collected = {}
    search_urls = get_search_urls(hashtag)

    for source_index, source_url in enumerate(search_urls, start=1):
        print(f"Buka sumber {source_index}/{len(search_urls)}: {source_url}")
        driver.get(source_url)
        time.sleep(6)

        for round_index in range(scroll_rounds):
            hrefs = extract_candidate_hrefs(driver)
            for href in hrefs:
                clean_url = normalize_facebook_url(href)
                if clean_url and is_candidate_post_url(clean_url):
                    collected[clean_url] = True

            print(
                f"Sumber {source_index} | Scroll {round_index + 1}/{scroll_rounds} | "
                f"link terkumpul: {len(collected)}"
            )
            if len(collected) >= target_count:
                break

            driver.execute_script("window.scrollBy(0, Math.max(window.innerHeight, 1200));")
            time.sleep(pause_seconds)

        if len(collected) >= target_count:
            break

    links = list(collected.keys())[:target_count]
    LINK_DEBUG_OUTPUT_PATH.write_text("\n".join(links), encoding="utf-8")
    print(f"Link debug disimpan ke: {LINK_DEBUG_OUTPUT_PATH}")
    return links


def is_video_post(driver):
    og_video = get_meta_content(driver, [
        'meta[property="og:video"]',
        'meta[property="og:video:url"]',
        'meta[property="og:video:secure_url"]',
    ])
    if og_video:
        return True

    videos = driver.find_elements(By.TAG_NAME, "video")
    return len(videos) > 0


def extract_first_photo_url(driver):
    meta_image = get_meta_content(driver, ['meta[property="og:image"]'])
    if meta_image:
        return meta_image

    images = driver.execute_script(
        """
        return Array.from(document.images).map(img => ({
            src: img.currentSrc || img.src || '',
            naturalWidth: img.naturalWidth || 0,
            naturalHeight: img.naturalHeight || 0,
            visible: !!(img.offsetWidth || img.offsetHeight || img.getClientRects().length)
        }));
        """
    )

    candidates = []
    for image in images:
        src = image.get("src") or ""
        width = image.get("naturalWidth") or 0
        height = image.get("naturalHeight") or 0
        if not src or src.startswith("data:"):
            continue
        if width < 300 or height < 300:
            continue
        if "emoji" in src.lower():
            continue
        candidates.append(image)

    if not candidates:
        return None

    candidates.sort(key=lambda item: (item.get("naturalWidth", 0) * item.get("naturalHeight", 0)), reverse=True)
    return candidates[0].get("src")


def extract_caption(driver):
    candidates = []
    xpaths = [
        "//div[@data-ad-preview='message']",
        "//div[contains(@class, 'userContent')]",
        "//div[@role='article']//div[@dir='auto']",
    ]
    for xpath in xpaths:
        for element in driver.find_elements(By.XPATH, xpath):
            text = clean_caption_text(element.text)
            if text and len(text) >= 5 and not re.search(r'\d{1,3}(?:[.,]\d{3})*(?:[.,]\d+)?\s*(?:likes?|reactions?)', text, flags=re.IGNORECASE):
                candidates.append(text)

    if candidates:
        unique_candidates = []
        for item in candidates:
            if item not in unique_candidates:
                unique_candidates.append(item)
        return max(unique_candidates, key=len)

    return None


def extract_date(driver):
    meta_date = get_meta_content(driver, ['meta[property="article:published_time"]'])
    if meta_date:
        return meta_date

    try:
        time_element = driver.find_element(By.XPATH, "//time[@datetime]")
        datetime_value = normalize_space(time_element.get_attribute("datetime"))
        if datetime_value:
            return datetime_value
    except NoSuchElementException:
        pass

    try:
        abbr = driver.find_element(By.XPATH, "//abbr[@data-utime]")
        data_utime = normalize_space(abbr.get_attribute("data-utime"))
        if data_utime:
            if data_utime.isdigit():
                return datetime.fromtimestamp(int(data_utime), tz=timezone.utc).isoformat()
            return data_utime
    except NoSuchElementException:
        pass

    html = driver.page_source or ""
    for pattern in [r'"publish_time":(\d{10})', r'"creation_time":(\d{10})']:
        match = re.search(pattern, html)
        if match:
            return datetime.fromtimestamp(int(match.group(1)), tz=timezone.utc).isoformat()

    return None


def _metric_to_int(raw_text):
    if raw_text is None:
        return None

    text = normalize_space(raw_text)
    if not text:
        return None

    lowered = text.lower()
    lowered = lowered.replace("orang menyukai ini", "")
    lowered = lowered.replace("people reacted to this", "")
    lowered = lowered.replace("people like this", "")
    lowered = lowered.replace("reactions", "")
    lowered = lowered.replace("reaction", "")
    lowered = lowered.replace("likes", "")
    lowered = lowered.replace("like", "")
    lowered = lowered.replace("suka", "")
    lowered = lowered.strip()

    match = re.search(r"(\d+(?:[.,]\d+)?)\s*(k|m|b|rb|jt)?", lowered)
    if not match:
        return None

    number_part = match.group(1)
    suffix = match.group(2)

    if suffix:
        number_value = float(number_part.replace(",", "."))
        multipliers = {
            "k": 1_000,
            "rb": 1_000,
            "m": 1_000_000,
            "jt": 1_000_000,
            "b": 1_000_000_000,
        }
        return int(number_value * multipliers[suffix])

    digits_only = re.sub(r"[^\d]", "", number_part)
    if not digits_only:
        return None
    return int(digits_only)


def clean_caption_text(text):
    cleaned = normalize_space(text)
    if not cleaned:
        return None

    # Buang pola header halaman yang sering ikut terbaca sebagai caption.
    cleaned = re.sub(r'^\s*[^.]{1,80}?\.\s*\d{1,3}(?:[.,]\d{3})*(?:[.,]\d+)?\s*(?:likes?|reactions?)\.?\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'^\s*\d{1,3}(?:[.,]\d{3})*(?:[.,]\d+)?\s*(?:likes?|reactions?)\.?\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(
        r'^\s*\d{1,3}(?:[.,]\d{3})*(?:[.,]\d+)?\s*(?:likes?|reactions?)\b[^A-Za-z0-9#@]*',
        '',
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(r'^\s*[^.]{1,80}?\.\s*', '', cleaned, count=1)
    cleaned = re.sub(r'\s*[·•]\s*\d+\s*talking about this\.?', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\bSee more\b\.?$', '', cleaned, flags=re.IGNORECASE)
    cleaned = normalize_space(cleaned)
    if not cleaned or len(cleaned) < 3:
        return None
    noisy_markers = [
        'likes', 'like this', 'talking about this', 'followers', 'follow', 'log into facebook',
        'kreator video', 'lawyer & law firm', 'member', 'wa.me',
    ]
    lowered = cleaned.lower()
    if any(marker in lowered for marker in noisy_markers):
        return None
    return cleaned


def normalize_image_url(image_url):
    if not image_url:
        return None
    parsed = urlparse(image_url)
    if not parsed.scheme or not parsed.netloc:
        return normalize_space(image_url)
    # Query token CDN sering berubah, jadi abaikan untuk deduplikasi.
    return urlunparse((parsed.scheme, parsed.netloc, parsed.path, '', '', ''))


def extract_like_from_text(raw_text):
    text = normalize_space(raw_text)
    if not text:
        return None

    patterns = [
        r'(\d+(?:[.,]\d+)?\s*(?:k|m|b|rb|jt)?)\s*(?:likes?|reactions?)',
        r'(\d+(?:[.,]\d+)?\s*(?:k|m|b|rb|jt)?)\s*orang menyukai ini',
        r'(\d+(?:[.,]\d+)?\s*(?:k|m|b|rb|jt)?)\s*people reacted to this',
        r'(\d+(?:[.,]\d+)?\s*(?:k|m|b|rb|jt)?)\s*people like this',
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return _metric_to_int(match.group(1))
    return None


def extract_like_count(driver):
    texts = driver.execute_script(
        """
        const nodes = Array.from(document.querySelectorAll('[aria-label], span, a, div'));
        const values = [];
        for (const node of nodes) {
            const aria = (node.getAttribute('aria-label') || '').trim();
            const text = (node.innerText || '').trim();
            if (aria) values.push(aria);
            if (text) values.push(text);
        }
        return Array.from(new Set(values));
        """
    )

    for text in texts:
        parsed = extract_like_from_text(text)
        if parsed is not None:
            return parsed

    fallback_candidates = [
        driver.title,
        get_meta_content(driver, ['meta[property="og:description"]', 'meta[name="description"]']),
    ]
    for candidate in fallback_candidates:
        parsed = extract_like_from_text(candidate)
        if parsed is not None:
            return parsed

    return None


def extract_location(driver):
    title_text = get_meta_content(driver, ['meta[property="og:title"]']) or driver.title
    title_text = normalize_space(title_text) or ""

    title_patterns = [
        r"\bis at\s+([^|]+)",
        r"\bberada di\s+([^|]+)",
        r"\bat\s+([^|]+)",
    ]
    for pattern in title_patterns:
        match = re.search(pattern, title_text, flags=re.IGNORECASE)
        if match:
            value = normalize_space(match.group(1))
            if value:
                return value

    location_xpaths = [
        "//a[contains(@href, '/maps/') or contains(@href, '/places/') or contains(@href, '/location/')]",
        "//a[contains(@href, 'google.com/maps')]",
    ]
    for xpath in location_xpaths:
        for element in driver.find_elements(By.XPATH, xpath):
            text = normalize_space(element.text)
            if text and len(text) >= 2:
                return text

    return None


def detect_image_extension(response, image_url):
    content_type = (response.headers.get("Content-Type") or "").lower()
    if "jpeg" in content_type or "jpg" in content_type:
        return ".jpg"
    if "png" in content_type:
        return ".png"
    if "webp" in content_type:
        return ".webp"

    path_suffix = Path(urlparse(image_url).path).suffix.lower()
    if path_suffix in {".jpg", ".jpeg", ".png", ".webp"}:
        return ".jpg" if path_suffix == ".jpeg" else path_suffix

    return ".jpg"


def download_image(session, image_url, file_stem):
    response = session.get(image_url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    extension = detect_image_extension(response, image_url)
    file_name = f"{file_stem}{extension}"
    file_path = IMAGE_OUTPUT_DIR / file_name
    file_path.write_bytes(response.content)
    image_hash = hashlib.sha1(response.content).hexdigest()
    return file_name, file_path, image_hash


def scrape_single_post(driver, post_url):
    driver.get(post_url)
    WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    time.sleep(POST_LOAD_WAIT_SECONDS)

    if is_video_post(driver):
        return None

    image_url = extract_first_photo_url(driver)
    if not image_url:
        return None

    caption = extract_caption(driver) or ""
    like_count = extract_like_count(driver)
    if like_count is None:
        like_count = extract_like_from_text(caption)
    date_value = extract_date(driver) or ""
    location = extract_location(driver) or "null"
    post_key = extract_post_identity_key(driver, post_url)
    image_key = normalize_image_url(image_url)

    return {
        "caption": caption,
        "like": like_count,
        "date": date_value,
        "location": location,
        "_post_key": post_key,
        "_image_url": image_url,
        "_image_key": image_key,
    }


def save_results_to_csv(rows, csv_path):
    if not rows:
        return
    df = pd.DataFrame(rows)
    ordered_columns = ["file_name", "caption", "like", "date", "location"]
    df = df[ordered_columns]
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

In [14]:
driver = None
results = []

try:
    driver = build_driver()
    wait_for_manual_login(driver)

    session = build_requests_session_from_driver(driver)
    post_links = collect_post_links(
        driver=driver,
        hashtag=HASHTAG,
        target_count=COLLECT_LINK_TARGET,
        scroll_rounds=SCROLL_ROUNDS,
        pause_seconds=SCROLL_PAUSE_SECONDS,
    )

    print(f"Total kandidat post: {len(post_links)}")

    saved_count = 0
    seen_post_keys = set()
    seen_image_urls = set()
    seen_signatures = set()
    seen_image_hashes = set()
    for loop_index, post_url in enumerate(post_links, start=1):
        if saved_count >= MAX_SAVED_POSTS:
            break

        print(f"[{loop_index}/{len(post_links)}] Proses: {post_url}")
        try:
            row = scrape_single_post(driver, post_url)
            if row is None:
                print("  -> Skip (video / tidak ada foto yang valid)")
                continue

            post_key = row.pop("_post_key", None)
            image_url = row.pop("_image_url", None)
            image_key = row.pop("_image_key", None)
            caption_key = normalize_space(row.get("caption", "")[:180].lower()) or ""
            signature = f"{caption_key}|{row.get('like', '')}|{row.get('date', '')}|{row.get('location', '')}"

            if post_key and post_key in seen_post_keys:
                print("  -> Skip (duplikat postingan yang sama)")
                continue

            if image_url and image_url in seen_image_urls:
                print("  -> Skip (duplikat foto yang sama)")
                continue

            if image_key and image_key in seen_image_urls:
                print("  -> Skip (duplikat foto canonical)")
                continue

            if signature in seen_signatures:
                print("  -> Skip (duplikat konten caption/date/lokasi)")
                continue

            file_stem = f"{saved_count + 1:03d}"
            file_name, file_path, image_hash = download_image(session, image_url, file_stem)
            if image_hash in seen_image_hashes:
                try:
                    file_path.unlink()
                except OSError:
                    pass
                print("  -> Skip (duplikat hash gambar)")
                continue
            row["file_name"] = file_name

            if post_key:
                seen_post_keys.add(post_key)
            if image_url:
                seen_image_urls.add(image_url)
            if image_key:
                seen_image_urls.add(image_key)
            seen_signatures.add(signature)
            seen_image_hashes.add(image_hash)

            results.append(row)
            saved_count += 1
            save_results_to_csv(results, CSV_OUTPUT_PATH)
            print(f"  -> Tersimpan: {row['file_name']}")
        except requests.HTTPError as exc:
            print(f"  -> Gagal download gambar: {exc}")
        except TimeoutException:
            print("  -> Timeout saat membuka post")
        except WebDriverException as exc:
            print(f"  -> Error Selenium: {exc}")
        except Exception as exc:
            print(f"  -> Error umum: {exc}")

    save_results_to_csv(results, CSV_OUTPUT_PATH)
    print("\nSelesai.")
    print(f"Foto tersimpan : {saved_count}")
    print(f"CSV metadata   : {CSV_OUTPUT_PATH}")

finally:
    if driver is not None:
        driver.quit()


Login ke Facebook di browser yang terbuka.
Setelah login selesai dan halaman Facebook sudah siap, kembali ke notebook lalu tekan Enter.
Buka sumber 1/3: https://www.facebook.com/hashtag/wisatalampungselatan
Sumber 1 | Scroll 1/35 | link terkumpul: 3
Sumber 1 | Scroll 2/35 | link terkumpul: 8
Sumber 1 | Scroll 3/35 | link terkumpul: 10
Sumber 1 | Scroll 4/35 | link terkumpul: 13
Sumber 1 | Scroll 5/35 | link terkumpul: 13
Sumber 1 | Scroll 6/35 | link terkumpul: 13
Sumber 1 | Scroll 7/35 | link terkumpul: 13
Sumber 1 | Scroll 8/35 | link terkumpul: 13
Sumber 1 | Scroll 9/35 | link terkumpul: 13
Sumber 1 | Scroll 10/35 | link terkumpul: 13
Sumber 1 | Scroll 11/35 | link terkumpul: 13
Sumber 1 | Scroll 12/35 | link terkumpul: 13
Sumber 1 | Scroll 13/35 | link terkumpul: 18
Sumber 1 | Scroll 14/35 | link terkumpul: 19
Sumber 1 | Scroll 15/35 | link terkumpul: 23
Sumber 1 | Scroll 16/35 | link terkumpul: 23
Sumber 1 | Scroll 17/35 | link terkumpul: 26
Sumber 1 | Scroll 18/35 | link terkumpu

In [15]:
if CSV_OUTPUT_PATH.exists():
    preview_df = pd.read_csv(CSV_OUTPUT_PATH)
    display(preview_df.head(10))
else:
    print("CSV belum terbentuk. Jalankan cell scraping terlebih dahulu.")

,file_name,caption,like,date,location
0,001.jpg,"Sekitar lampung, Bandar Lampung. 10,592 likes ...",10592.0,2024-05-15T13:40:00+00:00,NaN
1,002.jpg,"Benny Abriantoz Soza. 6,137 likes. Bersama Kaw...",6137.0,2026-01-06T09:26:35+00:00,NaN
2,003.jpg,Very nice,NaN,2023-11-22T16:22:29+00:00,NaN
3,004.jpg,❤😍 #fotofyp #bonus #pantaiklaralampung #wisata...,NaN,2025-02-11T09:15:54+00:00,NaN
4,005.jpg,"Dewi Niken Ariyanti. 5,407 likes Mom of 2 sons",5407.0,2024-09-13T07:39:43+00:00,NaN
5,006.jpg,NaN,NaN,2022-07-03T03:40:42+00:00,NaN
6,007.jpg,NaN,NaN,2022-01-09T07:15:58+00:00,NaN
7,008.jpg,"Law Office Triple ""A"" & Partners. 194 likes. L...",194.0,2023-08-16T11:48:10+00:00,NaN
8,009.jpg,"Abel Tira Ramadhan. 6,873 likes Riview,Tutoria...",6873.0,2025-04-06T14:10:16+00:00,NaN
9,010.jpg,#junglesealampung #wisatalampungselatan #jusju...,NaN,2026-01-27T04:26:34+00:00,NaN


## Kalau hasil selector belum pas

Facebook cukup sering mengubah struktur HTML. Kalau nanti ada field yang belum terambil dengan baik, bagian yang paling mungkin perlu penyesuaian adalah:

- fungsi `collect_post_links()`
- fungsi `extract_like_count()`
- fungsi `extract_location()`

Kalau Anda mau, setelah dicoba saya bisa bantu revisi notebook ini berdasarkan error atau contoh halaman yang muncul di browser Anda.
